In [1]:
import ee 
import geemap
import geopandas as gpd

import pprint as pp

ykf = gpd.read_file('./data/YKflats_roi_shape.shp')

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

Collection of Sentinel-2 Images on target date

In [2]:

roi_ee = ee.Geometry(ykf.iloc[0].geometry.__geo_interface__)

date = '2020-05-29'
date_plus1d = '2020-05-30' 
#Required to filter by date, but end_date is exclusive so not in image collection

s2_spec_reflec = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi_ee)
    .filterDate(date, date_plus1d)
)


# pp.pp(s2_spec_reflec.first().getInfo())
s2_total_imgs = s2_spec_reflec.size().getInfo()

s2_spec_reflec = s2_spec_reflec.map(lambda img: img.clip(roi_ee))
s2_spec_reflec = s2_spec_reflec.mosaic()


In [4]:
s2_cl_prob = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
    .filterBounds(roi_ee)
    .filterDate(date, date_plus1d)
)
s2_cl_prob = s2_cl_prob.map(lambda img: img.clip(roi_ee))
s2_cl_prob = s2_cl_prob.mosaic()

s2_cl_mask = s2_cl_prob.select('probability').lt(25).rename('cl_binary')

In [5]:
# Get the boundary of the Sentinel-2 clipped image
s2_data_mask = s2_spec_reflec.mask().reduce(ee.Reducer.anyNonZero())
s2_boundary = s2_data_mask.reduceToVectors(
    geometry=roi_ee,
    geometryType='polygon',
    scale=10,
    maxPixels=1e13
)

s2_boundary = ee.Feature(s2_boundary.toList(s2_boundary.size()).get(0))


Collection of Landsat Images on target date

In [6]:
def optical_rescale(img):
    "Only converts optical bands, thermal bands not included"
    img = img.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    )
    img = img.multiply(0.0000275).add(-0.2)

    return img


ls_spec_reflec = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                  .filterBounds(roi_ee)
                  .filterDate(date, date_plus1d)
)
ls_spec_reflec = ls_spec_reflec.map(lambda img: img.clip(roi_ee))

ls_qa = ls_spec_reflec.select('QA_PIXEL').mosaic()


ls_spec_reflec = ls_spec_reflec.map(optical_rescale)

ls_total_imgs = ls_spec_reflec.size().getInfo()
print(ls_total_imgs)

ls_spec_reflec = ls_spec_reflec.mosaic()


2


In [10]:
ls_cl_mask = ls_qa.bitwiseAnd(1 << 3).eq(0)
pp.pp(ls_cl_mask.getInfo())

{'type': 'Image',
 'bands': [{'id': 'QA_PIXEL',
            'data_type': {'type': 'PixelType',
                          'precision': 'int',
                          'min': 0,
                          'max': 1},
            'crs': 'EPSG:4326',
            'crs_transform': [1, 0, 0, 0, 1, 0]}]}


In [8]:
# Get the bounds of the Landsat image

ls_data_mask = ls_spec_reflec.mask().reduce(ee.Reducer.anyNonZero())
ls_boundary = ls_data_mask.reduceToVectors(
    geometry=roi_ee,
    geometryType='polygon',
    scale=10,
    maxPixels=1e13
)

ls_boundary = ee.Feature(ls_boundary.toList(ls_boundary.size()).get(2))

In [11]:
Map = geemap.Map()

s2_true_col_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000
}

s2_cloud_prob_params = {
    'bands': ['probability'],
    'min': 0,
    'max': 100
}

ls_true_col_params = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
    'min': 0,
    'max': 0.3
}


Map.addLayer(s2_spec_reflec, s2_true_col_params, f'S2 True Color Composite on {date}')
Map.addLayer(ls_spec_reflec, ls_true_col_params, f'LS8 True Color Composite on {date}')

#Map.addLayer(s2_cl_prob, s2_cloud_prob_params, 'S2 Cloud Probability')
Map.addLayer(s2_cl_mask, {'min': 0, 'max': 1}, 'S2 Cloud Mask')
Map.addLayer(ls_cl_mask, {'min': 0, 'max': 1}, 'LS Cloud Mask')




# Map.addLayer(s2_boundary, {'color': 'blue'}, 'S2 Boundary')
# Map.addLayer(ls_boundary, {'color': 'green'}, 'LS Boundary')
# Map.addLayer(roi_ee, {'color': 'red'}, 'ROI Outline')
Map.centerObject(roi_ee, zoom=10)
Map


Map(center=[66.51859072213907, -146.00018537875346], controls=(WidgetControl(options=['position', 'transparent…